<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">زمان ثابت نداریم؛ قرارداد اندازه‌گیری داریم</h1>
<p style="text-align:right">درس 90 از 92 · پیش از سریع‌ترکردن، چه چیزی را زمان بگیریم؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">82-performance</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-15/chapter-02/82-performance.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">تأخیر هر اجرا و نرخ کل تولید را با واحد و مخرج درست بسنجید؛ نمایش کم‌بیتی را با سرعت بیشتر یکی ندانید.</p><p style="text-align:right"><span class="phrase-lead" style="white-space:nowrap">پیش‌نیاز: حلقهٔ</span> واقعی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">generate</code>، تعداد <bdi dir="ltr">Token</bdi> تازه، <bdi dir="ltr">Batch</bdi> و اندازهٔ <bdi dir="ltr">Tensor</bdi>.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۶۵–۱۱۵ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر دو دنباله هرکدام چهار <bdi dir="ltr">Token</bdi> تازه بسازند، صورت کسر <bdi dir="ltr">Throughput</bdi> چهار است یا هشت؟ <bdi dir="ltr">Prompt</bdi> هم در آن حساب می‌شود؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
from statistics import median
import torch
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
from mini_gpt.performance import benchmark_generation,quantize_symmetric

torch.set_num_threads(1)
torch.manual_seed(7)
model = MiniGPT(ModelConfig(12,32,16,2,1,0.0))
prompt = torch.ones(2,6,dtype=torch.long)
measured = benchmark_generation(model,prompt,new_tokens=4,repeats=3,warmup=1)
print(measured)
print('Actual CPU timing of an untrained model; not an answer-quality experiment.')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">workload_summary(seconds, batch, new_tokens)</code> برای زمان‌های مثبتِ چند اجرای هم‌اندازه، <bdi dir="ltr">dict</bdi> با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">median_seconds</code>، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">total_generated</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">pooled_tokens_per_second</code> بدهد. هر اجرا <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">batch*new_tokens</code> <bdi dir="ltr">Token</bdi> تازه می‌سازد؛ نرخ تجمیعی، کل این <bdi dir="ltr">Token</bdi>ها تقسیم بر مجموع زمان‌هاست. آن را با میانهٔ نرخ اجراها یکی نگیرید.</p>
</div>

In [ ]:
def workload_summary(seconds, batch, new_tokens):
    # TODO: واحد و تعداد اجرای تکراری را نگه دارید
    return None

In [ ]:
def test_exercise():
    result = workload_summary([0.1,0.3],2,4)
    if result is None:
        return False
    assert math.isclose(result['median_seconds'],0.2)
    assert result['total_generated'] == 16
    assert math.isclose(result['pooled_tokens_per_second'],40.0)
    other = workload_summary([1.0,2.0,6.0],1,3)
    assert other == {'median_seconds':2.0,'total_generated':9,'pooled_tokens_per_second':1.0}
    assert workload_summary([2.0],3,2) == {'median_seconds':2.0,'total_generated':6,'pooled_tokens_per_second':3.0}
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: workload_summary')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط تعداد بیت قراردادی <bdi dir="ltr">Quantization</bdi> را از ۴ به ۸ ببرید؛ همان وزن‌ها را بازسازی کنید. هم خطا و هم اندازهٔ ذخیرهٔ واقعی را چاپ کنید؛ خروجی هر دو در این مثال <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">int8</code> است، نه بسته‌بندی واقعی چهاربیتی.</p>
</div>

In [ ]:
values = model.language_model_head.weight.detach()
for bits in (4,8):
    integer,scale,reconstructed = quantize_symmetric(values,bits)
    print('bits:',bits,'max error:',(reconstructed-values).abs().max().item(),
          'actual integer bytes:',integer.numel()*integer.element_size(),
          'scale stored separately:',scale)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">نسخهٔ خراب طول کل خروجی را تعداد <bdi dir="ltr">Token</bdi> تولیدشده می‌نامد و <bdi dir="ltr">Prompt</bdi> را دوباره می‌شمارد. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">generated_count(prompt_lengths, output_lengths)</code> مجموع اختلاف طول‌های متناظر را بدهد. دو فهرست هم‌اندازه‌اند و خروجی هر دنباله کوتاه‌تر از <bdi dir="ltr">Prompt</bdi> نیست؛ صفر <bdi dir="ltr">Token</bdi> تازه هم معتبر است.</p>
</div>

In [ ]:
prompt_lengths = [6,6]
output_lengths = [10,10]
print('wrong generated-token count:',sum(output_lengths))
print('input lengths:',prompt_lengths,'output lengths:',output_lengths)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def generated_count(prompt_lengths, output_lengths):
    # TODO: فقط ادامهٔ تولیدشده
    return None

In [ ]:
def test_repair():
    result = generated_count(prompt_lengths,output_lengths)
    if result is None:
        return False
    assert result == 8
    assert generated_count([3,8],[5,9]) == 3
    assert generated_count([4],[4]) == 0
    assert generated_count([],[]) == 0
    assert generated_count([6,6],[len(row) for row in measured['output_ids']]) == 8
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: generated_count')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">زمان از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">generate</code> واقعی و بدون <bdi dir="ltr">KV Cache</bdi> آمده است. آموزش‌ندیده‌بودن مدل، شمارش محاسبه را ساختگی نمی‌کند، اما اجازهٔ نتیجه‌گیری دربارهٔ کیفیت نمی‌دهد. هیچ آزمونی زمان ثابت یا سریع‌ترشدن قطعی را مطالبه نمی‌کند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">کدام بخش هزینه در این زمان نیست؟ چرا مدل با <bdi dir="ltr">Parameter</bdi>های کمتر یا عددهای کم‌بیت‌تر الزاماً در هر محیط تأخیر کمتری ندارد؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-15/chapter-02/82-performance.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/82-performance.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>